# 05 Graphs and Genome Assembly

This notebook introduces graph-based thinking for genome assembly.

It supports **Module 08: Graphs and Genome Assembly** and connects selected Rosalind problems such as `GRPH`, `TREE`, `LONG`, `DBRU`, `PCOV`, `GASM`, `GREP`, `ASMQ`, and selected `BA3` Textbook Track problems.

The goal is to show how biological sequence reconstruction can be framed using overlaps, k-mers, adjacency lists, de Bruijn graphs, Eulerian paths, and simple assembly-quality metrics.

## Learning Goals

After completing this notebook, a learner should be able to:

- represent sequence relationships as graphs;
- construct an overlap graph from DNA strings;
- generate k-mers from a sequence;
- reconstruct a string from an ordered genome path;
- construct a de Bruijn graph from a string;
- construct a de Bruijn graph from k-mers;
- understand why Eulerian paths matter for genome assembly;
- compute simple assembly quality metrics such as N50;
- connect graph algorithms to genome reconstruction.

## Connection to Original Rosalind Solutions

This notebook is connected to my original Rosalind solutions preserved under:

- `original_rosalind_tracks/bioinformatics_stronghold/`
- `original_rosalind_tracks/bioinformatics_textbook_track/`
- selected graph foundations in `original_rosalind_tracks/algorithmic_heights/`

The notebook uses teaching-oriented implementations of graph and genome assembly concepts. The original solution files remain the solved-work archive, while this notebook reorganizes selected ideas for explanation, learning, and future reuse.

## 1. Why genome assembly is a graph problem

In genome assembly, we often have many short reads or k-mers and want to reconstruct a longer sequence.

Graphs are useful because:

- sequences can become nodes;
- overlaps can become edges;
- k-mers can become paths;
- reconstruction can become a graph traversal problem.

This is one of the strongest bridges between computer science graph algorithms and computational biology.

## 2. k-mer generation

A k-mer is a substring of length `k`.

This connects to Textbook Track problems such as `BA3A`.

In [ ]:
def generate_kmers(sequence: str, k: int) -> list[str]:
    """Return all k-mers of length k from a sequence."""
    if k <= 0:
        raise ValueError("k must be positive")

    return [sequence[i:i + k] for i in range(len(sequence) - k + 1)]


sequence = "AAGATTCTCTAAGA"
generate_kmers(sequence, k=4)

## 3. Genome path reconstruction

This connects to Textbook Track problem `BA3B`.

If k-mers are already ordered as a path, the original string can be reconstructed by taking the first k-mer and then appending the last character of each next k-mer.

In [ ]:
def reconstruct_from_path(kmers: list[str]) -> str:
    """Reconstruct a string from an ordered list of overlapping k-mers."""
    if not kmers:
        return ""

    reconstructed = kmers[0]

    for kmer in kmers[1:]:
        reconstructed += kmer[-1]

    return reconstructed


path = ["ACCGA", "CCGAA", "CGAAG", "GAAGC", "AAGCT"]
reconstruct_from_path(path)

## 4. Overlap graph

This connects to Rosalind problem `GRPH`.

An overlap graph connects two sequences if the suffix of one sequence overlaps with the prefix of another sequence.

For Rosalind `GRPH`, the usual overlap length is 3.

In [ ]:
def suffix(sequence: str, length: int) -> str:
    """Return the suffix of sequence with the given length."""
    return sequence[-length:]


def prefix(sequence: str, length: int) -> str:
    """Return the prefix of sequence with the given length."""
    return sequence[:length]


def overlap_graph(records: dict[str, str], overlap_length: int = 3) -> list[tuple[str, str]]:
    """Return directed edges for an overlap graph."""
    edges = []

    for source_id, source_sequence in records.items():
        for target_id, target_sequence in records.items():
            if source_id == target_id:
                continue

            if suffix(source_sequence, overlap_length) == prefix(target_sequence, overlap_length):
                edges.append((source_id, target_id))

    return edges


records = {
    "Rosalind_0498": "AAATAAA",
    "Rosalind_2391": "AAATTTT",
    "Rosalind_2323": "TTTTCCC",
    "Rosalind_0442": "AAATCCC",
    "Rosalind_5013": "GGGTGGG",
}

overlap_graph(records, overlap_length=3)

## 5. Adjacency list representation

Graphs are often stored as adjacency lists.

This representation will be reused for de Bruijn graphs.

In [ ]:
def edges_to_adjacency_list(edges: list[tuple[str, str]]) -> dict[str, list[str]]:
    """Convert a list of directed edges into an adjacency list."""
    adjacency = {}

    for source, target in edges:
        if source not in adjacency:
            adjacency[source] = []

        adjacency[source].append(target)

    return adjacency


edges = overlap_graph(records, overlap_length=3)
edges_to_adjacency_list(edges)

## 6. de Bruijn graph from a string

This connects to Textbook Track problem `BA3D`.

A de Bruijn graph of order `k` represents each k-mer as an edge from its `(k-1)`-prefix to its `(k-1)`-suffix.

In [ ]:
def de_bruijn_from_string(sequence: str, k: int) -> dict[str, list[str]]:
    """Construct a de Bruijn graph from a sequence and k-mer length."""
    graph = {}

    for kmer in generate_kmers(sequence, k):
        left = kmer[:-1]
        right = kmer[1:]

        if left not in graph:
            graph[left] = []

        graph[left].append(right)

    return graph


de_bruijn_from_string("AAGATTCTCTAAGA", k=4)

## 7. de Bruijn graph from k-mers

This connects to Rosalind problem `DBRU` and Textbook Track problem `BA3E`.

Instead of generating k-mers from one sequence, we may be given a collection of k-mers directly.

In [ ]:
def de_bruijn_from_kmers(kmers: list[str]) -> dict[str, list[str]]:
    """Construct a de Bruijn graph from a list of k-mers."""
    graph = {}

    for kmer in kmers:
        left = kmer[:-1]
        right = kmer[1:]

        if left not in graph:
            graph[left] = []

        graph[left].append(right)

    return graph


kmers = ["GAGG", "CAGG", "GGGG", "GGGA", "CAGG", "AGGG", "GGAG"]
de_bruijn_from_kmers(kmers)

## 8. Formatting adjacency lists

Rosalind graph problems often expect output in a readable adjacency-list format.

For example:

`A -> B,C,D`

In [ ]:
def format_adjacency_list(graph: dict[str, list[str]]) -> list[str]:
    """Return a sorted adjacency-list representation."""
    lines = []

    for source in sorted(graph):
        targets = sorted(graph[source])
        lines.append(f"{source} -> {','.join(targets)}")

    return lines


for line in format_adjacency_list(de_bruijn_from_kmers(kmers)):
    print(line)

## 9. Node balance and Eulerian paths

Genome reconstruction from de Bruijn graphs is related to finding an Eulerian path.

An Eulerian path uses every edge exactly once.

For a directed graph:

- a balanced node has equal in-degree and out-degree;
- the path start often has out-degree one greater than in-degree;
- the path end often has in-degree one greater than out-degree.

In [ ]:
from collections import defaultdict


def node_degrees(graph: dict[str, list[str]]) -> dict[str, tuple[int, int]]:
    """Return in-degree and out-degree for each node in a directed graph."""
    in_degree = defaultdict(int)
    out_degree = defaultdict(int)

    nodes = set(graph.keys())

    for source, targets in graph.items():
        out_degree[source] += len(targets)

        for target in targets:
            in_degree[target] += 1
            nodes.add(target)

    return {
        node: (in_degree[node], out_degree[node])
        for node in sorted(nodes)
    }


graph = de_bruijn_from_kmers(["CTTA", "ACCA", "TACC", "GGCT", "GCTT", "TTAC"])
node_degrees(graph)

## 10. Eulerian path using Hierholzer's algorithm

This connects conceptually to `BA3F`, `BA3G`, and `BA3H`.

The implementation below is intended for teaching. It assumes the input graph has an Eulerian path.

In [ ]:
def find_start_node(graph: dict[str, list[str]]) -> str:
    """Find a likely start node for an Eulerian path."""
    degrees = node_degrees(graph)

    for node, (in_deg, out_deg) in degrees.items():
        if out_deg - in_deg == 1:
            return node

    return next(iter(graph))


def eulerian_path(graph: dict[str, list[str]]) -> list[str]:
    """Return an Eulerian path for a directed graph using Hierholzer's algorithm."""
    graph_copy = {node: targets[:] for node, targets in graph.items()}

    for node in list(node_degrees(graph_copy).keys()):
        graph_copy.setdefault(node, [])

    start = find_start_node(graph_copy)
    stack = [start]
    path = []

    while stack:
        current = stack[-1]

        if graph_copy[current]:
            stack.append(graph_copy[current].pop())
        else:
            path.append(stack.pop())

    return path[::-1]


path_graph = de_bruijn_from_kmers(["CTTA", "ACCA", "TACC", "GGCT", "GCTT", "TTAC"])
eulerian_path(path_graph)

## 11. Reconstructing a sequence from an Eulerian path

This connects to Textbook Track problem `BA3H`.

Once an Eulerian path through `(k-1)`-mers is found, the sequence can be reconstructed by appending the last character of each next node.

In [ ]:
def reconstruct_from_node_path(path: list[str]) -> str:
    """Reconstruct a sequence from an ordered path of overlapping nodes."""
    if not path:
        return ""

    sequence = path[0]

    for node in path[1:]:
        sequence += node[-1]

    return sequence


node_path = eulerian_path(path_graph)
reconstruct_from_node_path(node_path)

## 12. Assembly quality: N50

This connects to Rosalind problem `ASMQ`.

N50 is a common assembly statistic. Given contig lengths, N50 is the contig length such that contigs of that length or longer cover at least 50% of the total assembly length.

In [ ]:
def n50(contig_lengths: list[int]) -> int:
    """Return the N50 value for a list of contig lengths."""
    if not contig_lengths:
        return 0

    sorted_lengths = sorted(contig_lengths, reverse=True)
    total_length = sum(sorted_lengths)
    threshold = total_length / 2

    running_sum = 0

    for length in sorted_lengths:
        running_sum += length

        if running_sum >= threshold:
            return length

    return 0


contigs = [500, 400, 300, 200, 100]
n50(contigs)

## 13. N75

N75 is similar to N50, but the threshold is 75% of the total assembly length.

In [ ]:
def nx(contig_lengths: list[int], x: float) -> int:
    """Return the Nx statistic for a list of contig lengths and percentage x."""
    if not contig_lengths:
        return 0

    sorted_lengths = sorted(contig_lengths, reverse=True)
    total_length = sum(sorted_lengths)
    threshold = total_length * (x / 100)

    running_sum = 0

    for length in sorted_lengths:
        running_sum += length

        if running_sum >= threshold:
            return length

    return 0


print("N50:", nx(contigs, 50))
print("N75:", nx(contigs, 75))

## 14. Mini exercise set

Try modifying the functions above to solve these small exercises.

1. Generate all 5-mers from a DNA sequence.
2. Build an overlap graph with overlap length 4 instead of 3.
3. Modify `format_adjacency_list()` so it preserves insertion order instead of sorting.
4. Build a de Bruijn graph from your own list of k-mers.
5. Compute in-degree and out-degree for every node in a graph.
6. Use the Eulerian path function to reconstruct a sequence from k-mers.
7. Compute N50 and N75 for a new list of contig lengths.

## Summary

This notebook showed how graph algorithms support genome assembly.

| Concept | Genome Assembly Use |
|---|---|
| k-mers | short sequence fragments |
| Genome path | ordered reconstruction from overlapping k-mers |
| Overlap graph | read-to-read overlap representation |
| de Bruijn graph | compact k-mer transition representation |
| Eulerian path | reconstruction using every k-mer edge |
| Adjacency list | graph storage format |
| N50/N75 | assembly quality summaries |

These ideas prepare the learner for later work in genome indexing, graph-based biological data representation, phylogenetics, and ML-ready computational biology workflows.